### **Sincronía audiovisual y grounding temporal**

#### **Desfase, alineamiento no lineal, localización de evidencia e identificabilidad**

Este cuaderno separa tres problemas que suelen confundirse:

1. Correspondencia semántica entre audio y video.
2. Sincronización temporal precisa.
3. Grounding de un evento dentro de un intervalo.

La pregunta central es:

> ¿Cómo medir si dos modalidades describen el mismo evento y, además, ocurren en el mismo instante?


### **Preguntas de investigación**

#### **Hipótesis de trabajo**

**H1.** La correspondencia semántica puede permanecer alta aunque exista un desfase temporal.

**H2.** La correlación cruzada recupera desplazamientos constantes cuando la señal tiene estructura suficiente.

**H3.** Dynamic Time Warping supera al alineamiento directo cuando existe cambio de velocidad.

**H4.** Los eventos repetidos producen múltiples máximos plausibles y reducen identificabilidad.

**H5.** Una respuesta correcta puede coexistir con un intervalo de grounding incorrecto.


### **Configuración reproducible**

#### **Importaciones, rutas y semilla**


In [ ]:
from __future__ import annotations

import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def set_seed(seed: int) -> int:
    """Fija las semillas utilizadas en el cuaderno."""
    random.seed(seed)
    np.random.seed(seed)
    print("Semilla fijada:", seed)
    return seed

SEED = set_seed(226)
RESULTS_DIR = Path("resultados/cuaderno26_mcc225")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Directorio de resultados:", RESULTS_DIR)


### **Metadatos del experimento**

#### **Registro del protocolo**


In [ ]:
@dataclass
class ExperimentMetadata:
    course: str
    week: str
    notebook: str
    topic: str
    seed: int
    execution_mode: str


metadata = ExperimentMetadata(
    course="MCC225",
    week="Semana 12",
    notebook="Cuaderno26-MCC225",
    topic="Sincronía audiovisual y grounding temporal",
    seed=SEED,
    execution_mode="CPU con señales controladas",
)

with open(RESULTS_DIR / "metadatos.json", "w", encoding="utf-8") as file:
    json.dump(asdict(metadata), file, indent=2, ensure_ascii=False)

asdict(metadata)


### **Marco formal**

#### **Desfase y localización temporal**

Para dos secuencias $a_t$ y $v_t$, el desfase se estima mediante:

$$
\hat{\delta}
=
\operatorname*{argmax}_{\delta} C(\delta).
$$

Para grounding temporal, el sistema predice un intervalo:

$$
\hat{I} = [\hat{t}_s, \hat{t}_e].
$$

La calidad del grounding puede evaluarse mediante:

$$
\operatorname{tIoU}(I, \hat{I})
=
\frac{|I \cap \hat{I}|}{|I \cup \hat{I}|}.
$$


### **Generación de señales controladas**

#### **Eventos, desfases, ruido y repetición**


In [ ]:
def generate_event_sequence(
    length: int,
    event_start: int,
    event_end: int,
    noise_level: float,
    seed: int,
    repeated_start: int | None = None,
) -> np.ndarray:
    """Genera una secuencia con uno o dos eventos suaves."""
    rng = np.random.default_rng(seed)
    sequence = rng.normal(0.0, noise_level, size=length)
    event_length = event_end - event_start
    event_shape = np.hanning(event_length + 2)[1:-1]
    sequence[event_start:event_end] += event_shape

    if repeated_start is not None:
        repeated_end = repeated_start + event_length
        sequence[repeated_start:repeated_end] += event_shape

    return sequence.astype(np.float64)


def shift_sequence(sequence: np.ndarray, offset: int) -> np.ndarray:
    """Desplaza una secuencia y completa con ceros."""
    shifted = np.zeros_like(sequence)
    if offset > 0:
        shifted[offset:] = sequence[:-offset]
    elif offset < 0:
        shifted[:offset] = sequence[-offset:]
    else:
        shifted[:] = sequence
    return shifted


def resample_sequence(sequence: np.ndarray, factor: float) -> np.ndarray:
    """Modifica la velocidad de una secuencia mediante interpolación lineal."""
    new_length = max(2, int(round(len(sequence) / factor)))
    source_positions = np.linspace(0.0, 1.0, len(sequence))
    target_positions = np.linspace(0.0, 1.0, new_length)
    return np.interp(target_positions, source_positions, sequence)


reference = generate_event_sequence(
    length=64,
    event_start=18,
    event_end=30,
    noise_level=0.02,
    seed=SEED,
)

plt.figure(figsize=(9, 3))
plt.plot(reference)
plt.xlabel("Índice temporal")
plt.ylabel("Amplitud")
plt.title("Evento temporal de referencia")
plt.tight_layout()
plt.show()


### **Correlación cruzada**

#### **Estimación de desfases constantes**


In [ ]:
def normalized_cross_correlation(
    reference_sequence: np.ndarray,
    observed_sequence: np.ndarray,
    max_offset: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Calcula correlación normalizada para un rango de desfases."""
    offsets = np.arange(-max_offset, max_offset + 1)
    scores = []

    for offset in offsets:
        aligned = shift_sequence(observed_sequence, -int(offset))
        first = reference_sequence - reference_sequence.mean()
        second = aligned - aligned.mean()
        denominator = float(np.linalg.norm(first) * np.linalg.norm(second))
        score = 0.0 if denominator == 0.0 else float(first @ second / denominator)
        scores.append(score)

    return offsets, np.asarray(scores)


def estimate_time_offset(
    reference_sequence: np.ndarray,
    observed_sequence: np.ndarray,
    max_offset: int,
) -> tuple[int, float]:
    """Estima el desfase que maximiza la correlación normalizada."""
    offsets, scores = normalized_cross_correlation(
        reference_sequence,
        observed_sequence,
        max_offset,
    )
    best_index = int(np.argmax(scores))
    return int(offsets[best_index]), float(scores[best_index])


offset_rows = []
correlation_curves = {}
for true_offset in [-12, -8, -4, 0, 4, 8, 12]:
    observed = shift_sequence(reference, true_offset)
    predicted_offset, best_score = estimate_time_offset(
        reference,
        observed,
        max_offset=16,
    )
    offsets, scores = normalized_cross_correlation(reference, observed, max_offset=16)
    correlation_curves[true_offset] = (offsets, scores)
    offset_rows.append({
        "desfase_real": true_offset,
        "desfase_estimado": predicted_offset,
        "error_absoluto": abs(true_offset - predicted_offset),
        "correlacion_maxima": best_score,
    })

offset_table = pd.DataFrame(offset_rows)
offset_table


In [ ]:
plt.figure(figsize=(10, 4))
for true_offset in [-8, 0, 8]:
    offsets, scores = correlation_curves[true_offset]
    plt.plot(offsets, scores, label=f"Desfase real {true_offset}")

plt.xlabel("Desfase evaluado")
plt.ylabel("Correlación normalizada")
plt.title("Curvas de correlación cruzada")
plt.legend()
plt.tight_layout()
plt.show()


### **Robustez frente a ruido**

#### **Múltiples semillas y tolerancia temporal**


In [ ]:
def offset_mean_absolute_error(
    true_offsets: np.ndarray,
    predicted_offsets: np.ndarray,
) -> float:
    """Calcula el error absoluto medio del desfase."""
    return float(np.mean(np.abs(true_offsets - predicted_offsets)))


def tolerance_accuracy(
    true_offsets: np.ndarray,
    predicted_offsets: np.ndarray,
    tolerance: int,
) -> float:
    """Calcula la proporción de estimaciones dentro de una tolerancia."""
    errors = np.abs(true_offsets - predicted_offsets)
    return float(np.mean(errors <= tolerance))


noise_rows = []
for noise_level in [0.0, 0.05, 0.10, 0.20, 0.35]:
    for current_seed in range(30):
        rng = np.random.default_rng(1000 + current_seed)
        true_offset = int(rng.integers(-10, 11))
        clean_observed = shift_sequence(reference, true_offset)
        noisy_observed = clean_observed + rng.normal(
            0.0,
            noise_level,
            size=len(clean_observed),
        )
        predicted_offset, best_score = estimate_time_offset(
            reference,
            noisy_observed,
            max_offset=14,
        )
        noise_rows.append({
            "nivel_ruido": noise_level,
            "semilla": current_seed,
            "desfase_real": true_offset,
            "desfase_estimado": predicted_offset,
            "error_absoluto": abs(true_offset - predicted_offset),
            "correlacion_maxima": best_score,
        })

noise_table = pd.DataFrame(noise_rows)
noise_summary = noise_table.groupby("nivel_ruido", as_index=False).agg(
    error_medio=("error_absoluto", "mean"),
    correlacion_media=("correlacion_maxima", "mean"),
)
noise_summary["exactitud_tolerancia_1"] = [
    tolerance_accuracy(
        group["desfase_real"].to_numpy(),
        group["desfase_estimado"].to_numpy(),
        tolerance=1,
    )
    for _, group in noise_table.groupby("nivel_ruido")
]
noise_summary


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(noise_summary["nivel_ruido"], noise_summary["error_medio"], marker="o")
plt.xlabel("Nivel de ruido")
plt.ylabel("Error absoluto medio")
plt.title("Robustez de la estimación de desfase")
plt.tight_layout()
plt.show()


### **Dynamic Time Warping**

#### **Alineamiento bajo cambios de velocidad**

Dynamic Time Warping permite repetir o saltar posiciones para alinear secuencias con deformación temporal. Su costo clásico es cuadrático en las longitudes de las secuencias.


In [ ]:
def dynamic_time_warping(
    first_sequence: np.ndarray,
    second_sequence: np.ndarray,
) -> tuple[float, np.ndarray]:
    """Calcula la distancia DTW y la matriz acumulada."""
    first_length = len(first_sequence)
    second_length = len(second_sequence)
    accumulated = np.full((first_length + 1, second_length + 1), np.inf)
    accumulated[0, 0] = 0.0

    for first_index in range(1, first_length + 1):
        for second_index in range(1, second_length + 1):
            local_cost = abs(
                first_sequence[first_index - 1]
                - second_sequence[second_index - 1]
            )
            accumulated[first_index, second_index] = local_cost + min(
                accumulated[first_index - 1, second_index],
                accumulated[first_index, second_index - 1],
                accumulated[first_index - 1, second_index - 1],
            )

    normalized_distance = float(
        accumulated[first_length, second_length]
        / (first_length + second_length)
    )
    return normalized_distance, accumulated


def direct_resampled_distance(
    first_sequence: np.ndarray,
    second_sequence: np.ndarray,
) -> float:
    """Compara dos secuencias después de llevarlas a la misma longitud."""
    source_positions = np.linspace(0.0, 1.0, len(second_sequence))
    target_positions = np.linspace(0.0, 1.0, len(first_sequence))
    resized = np.interp(target_positions, source_positions, second_sequence)
    return float(np.mean(np.abs(first_sequence - resized)))


speed_rows = []
for speed_factor in [0.65, 0.80, 1.00, 1.20, 1.50]:
    modified = resample_sequence(reference, speed_factor)
    direct_distance = direct_resampled_distance(reference, modified)
    dtw_distance, _ = dynamic_time_warping(reference, modified)
    speed_rows.append({
        "factor_velocidad": speed_factor,
        "longitud_observada": len(modified),
        "distancia_directa": direct_distance,
        "distancia_dtw": dtw_distance,
    })

speed_table = pd.DataFrame(speed_rows)
speed_table


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    speed_table["factor_velocidad"],
    speed_table["distancia_directa"],
    marker="o",
    label="Alineamiento directo",
)
plt.plot(
    speed_table["factor_velocidad"],
    speed_table["distancia_dtw"],
    marker="o",
    label="DTW",
)
plt.xlabel("Factor de velocidad")
plt.ylabel("Distancia normalizada")
plt.title("Comparación bajo cambio de velocidad")
plt.legend()
plt.tight_layout()
plt.show()


### **Grounding temporal**

#### **Localización de un evento mediante similitud por ventana**

Se construyen features multidimensionales. El evento ocupa un intervalo conocido y la consulta representa el patrón que debe localizarse.


In [ ]:
@dataclass
class TemporalInterval:
    start: int
    end: int

    def duration(self) -> int:
        """Calcula la duración del intervalo."""
        return max(0, self.end - self.start)


def generate_feature_sequence(
    length: int,
    dimension: int,
    event_interval: TemporalInterval,
    query: np.ndarray,
    noise_level: float,
    seed: int,
) -> np.ndarray:
    """Genera features temporales con un evento alineado con una consulta."""
    rng = np.random.default_rng(seed)
    sequence = rng.normal(0.0, noise_level, size=(length, dimension))
    sequence[event_interval.start:event_interval.end] += query
    return sequence


def localize_event(
    sequence: np.ndarray,
    query: np.ndarray,
    window_size: int,
) -> tuple[TemporalInterval, np.ndarray]:
    """Localiza la ventana con mayor similitud media respecto de la consulta."""
    normalized_query = query / np.clip(np.linalg.norm(query), 1e-12, None)
    scores = []
    for start in range(0, len(sequence) - window_size + 1):
        window = sequence[start:start + window_size]
        window_embedding = window.mean(axis=0)
        window_embedding = window_embedding / np.clip(
            np.linalg.norm(window_embedding),
            1e-12,
            None,
        )
        scores.append(float(window_embedding @ normalized_query))

    best_start = int(np.argmax(scores))
    return TemporalInterval(best_start, best_start + window_size), np.asarray(scores)


def temporal_iou(
    first_interval: TemporalInterval,
    second_interval: TemporalInterval,
) -> float:
    """Calcula Intersection over Union temporal."""
    intersection_start = max(first_interval.start, second_interval.start)
    intersection_end = min(first_interval.end, second_interval.end)
    intersection = max(0, intersection_end - intersection_start)
    union = first_interval.duration() + second_interval.duration() - intersection
    return 0.0 if union == 0 else float(intersection / union)


def boundary_errors(
    predicted_interval: TemporalInterval,
    reference_interval: TemporalInterval,
) -> dict[str, float]:
    """Calcula errores de inicio, final, centro y duración."""
    predicted_center = (predicted_interval.start + predicted_interval.end) / 2.0
    reference_center = (reference_interval.start + reference_interval.end) / 2.0
    return {
        "error_inicio": abs(predicted_interval.start - reference_interval.start),
        "error_final": abs(predicted_interval.end - reference_interval.end),
        "error_centro": abs(predicted_center - reference_center),
        "error_duracion": abs(
            predicted_interval.duration() - reference_interval.duration()
        ),
    }


query = np.array([0.8, 0.3, 0.6, 0.1], dtype=np.float64)
query = query / np.linalg.norm(query)
reference_interval = TemporalInterval(18, 30)
feature_sequence = generate_feature_sequence(
    length=64,
    dimension=4,
    event_interval=reference_interval,
    query=query,
    noise_level=0.18,
    seed=SEED,
)
predicted_interval, grounding_scores = localize_event(
    feature_sequence,
    query,
    window_size=12,
)

grounding_result = {
    "intervalo_real": [reference_interval.start, reference_interval.end],
    "intervalo_estimado": [predicted_interval.start, predicted_interval.end],
    "tiou": temporal_iou(predicted_interval, reference_interval),
    **boundary_errors(predicted_interval, reference_interval),
}
grounding_result


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(grounding_scores, label="Puntaje por ventana")
plt.axvline(reference_interval.start, linestyle="--", label="Inicio real")
plt.axvline(predicted_interval.start, linestyle=":", label="Inicio estimado")
plt.xlabel("Inicio de ventana")
plt.ylabel("Similitud")
plt.title("Localización temporal del evento")
plt.legend()
plt.tight_layout()
plt.show()


### **Identificabilidad y eventos repetidos**

#### **Múltiples alineamientos plausibles**

Cuando la señal contiene el mismo evento en dos instantes, la correlación puede producir varios máximos. En ese caso, un único desfase no está completamente identificado por la evidencia.


In [ ]:
repeated_reference = generate_event_sequence(
    length=80,
    event_start=12,
    event_end=22,
    repeated_start=48,
    noise_level=0.01,
    seed=SEED,
)
repeated_observed = shift_sequence(repeated_reference, 5)
repeated_offsets, repeated_scores = normalized_cross_correlation(
    repeated_reference,
    repeated_observed,
    max_offset=30,
)

best_indices = np.argsort(-repeated_scores)[:5]
ambiguity_table = pd.DataFrame({
    "desfase_candidato": repeated_offsets[best_indices],
    "correlacion": repeated_scores[best_indices],
})
ambiguity_table


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(repeated_offsets, repeated_scores)
plt.xlabel("Desfase evaluado")
plt.ylabel("Correlación")
plt.title("Ambigüedad producida por eventos repetidos")
plt.tight_layout()
plt.show()


### **Incertidumbre mediante bootstrap**

#### **Intervalo para el error de desfase**


In [ ]:
def bootstrap_mean_interval(
    values: np.ndarray,
    repetitions: int,
    confidence_level: float,
    seed: int,
) -> tuple[float, float]:
    """Calcula un intervalo bootstrap para la media."""
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(repetitions):
        sample = rng.choice(values, size=len(values), replace=True)
        means.append(float(np.mean(sample)))
    alpha = 1.0 - confidence_level
    return (
        float(np.quantile(means, alpha / 2.0)),
        float(np.quantile(means, 1.0 - alpha / 2.0)),
    )


error_interval = bootstrap_mean_interval(
    noise_table["error_absoluto"].to_numpy(),
    repetitions=2000,
    confidence_level=0.95,
    seed=SEED,
)
print("Error absoluto medio:", float(noise_table["error_absoluto"].mean()))
print("Intervalo bootstrap del 95 por ciento:", error_interval)


### **Protocolo de evaluación**

#### **Métricas mínimas**

Un estudio de sincronía y grounding debe reportar:

1. Error absoluto del desfase.
2. Exactitud dentro de una tolerancia.
3. Correlación máxima.
4. Distancia DTW normalizada.
5. Intersection over Union temporal.
6. Error de inicio y final.
7. Error del centro y duración.
8. Desempeño por nivel de ruido.
9. Desempeño por factor de velocidad.
10. Casos ambiguos con múltiples alineamientos.
11. Intervalos de confianza.


### **Amenazas a la validez**

#### **Límites de la evidencia experimental**

1. Las señales sintéticas no representan reverberación, oclusión ni fuentes sonoras múltiples.
2. La correlación supone una relación aproximadamente lineal entre ambas secuencias.
3. DTW puede encontrar alineamientos numéricos sin significado causal.
4. La ventana de grounding tiene la misma duración que el evento real.
5. Los eventos repetidos muestran que la sincronía puede no ser identificable sin contexto adicional.
6. Una alta correspondencia semántica no demuestra sincronía precisa.


### **Conclusiones**

#### **Respuesta a la pregunta central**

La correspondencia y la sincronía son propiedades distintas. La correlación cruzada es adecuada para desfases constantes, mientras que DTW tolera cambios de velocidad. El grounding requiere métricas de intervalos y análisis de límites. Los eventos repetidos revelan que una solución puede ser ambigua incluso cuando la correlación es alta.


#### **Preguntas a desarrollar**


1. ¿Qué diferencia existe entre correspondencia semántica y sincronía?
2. ¿Cuándo falla la correlación cruzada?
3. ¿Qué tipo de deformación puede absorber DTW?
4. ¿Por qué DTW puede producir alineamientos sin significado?
5. ¿Qué limitaciones presenta Intersection over Union temporal?
6. ¿Cómo se detecta que existen varios desfases plausibles?
7. ¿Qué información adicional permitiría resolver un evento repetido?
8. ¿Cómo cambiaría el protocolo para evaluación en milisegundos?.


### **Referencias principales**

#### **Ruta de lectura**

1. Arandjelovic y Zisserman. **Look, Listen and Learn**.
2. Shi et al. **AV-HuBERT: Self-Supervised Learning of Audio-Visual Speech Representation**.
3. Lei et al. **QVHighlights: Detecting Moments and Highlights in Videos via Natural Language Queries**.
4. Cai et al. **TemporalBench: Benchmarking Fine-grained Temporal Understanding for Multimodal Video Models**.
5. Zhou et al. **AV-SyncBench: Decoupled Benchmarking of Temporal and Semantic Audio-Visual Synchronization**.


### **Exportación de resultados**

#### **Evidencia reproducible**


In [ ]:
offset_table.to_csv(
    RESULTS_DIR / "resultados_desfase.csv",
    index=False,
    encoding="utf-8",
)
noise_table.to_csv(
    RESULTS_DIR / "robustez_ruido.csv",
    index=False,
    encoding="utf-8",
)
speed_table.to_csv(
    RESULTS_DIR / "cambios_velocidad.csv",
    index=False,
    encoding="utf-8",
)
ambiguity_table.to_csv(
    RESULTS_DIR / "alineamientos_ambiguos.csv",
    index=False,
    encoding="utf-8",
)

summary = {
    "grounding": grounding_result,
    "error_desfase_medio": float(noise_table["error_absoluto"].mean()),
    "intervalo_bootstrap_95": list(error_interval),
}
with open(RESULTS_DIR / "resumen.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

print("Resultados exportados en:", RESULTS_DIR)
